In [ ]:
import sys
import os
import torch
project_root=r"your_path\Project"
sys.path.insert(0, project_root)
sys.path.append(r"your_path\Project\models")
sys.path.append(r"your_path\Project\utils")
sys.path.append(r"your_path\Project\dataset")

from shallow_autoencoder import ShallowAutoencoder
from deep_autoencoder import DeepAutoencoder
from residual_autoencoder import ResidualAutoencoder
from config import DEVICE
print(f"DEVICE: {DEVICE}")

DEVICE: cuda


In [2]:
model = ShallowAutoencoder().to(DEVICE)
print(model)

ShallowAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (bottleneck): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
    (5): ReLU(inplace=True)
    (6): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(32

In [8]:
model2 = DeepAutoencoder().to(DEVICE)
print(model2)

DeepAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (bottleneck): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2

In [3]:
model3 = ResidualAutoencoder().to(DEVICE)
print(model3)

ResidualAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (relu): ReLU(inplace=True)
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (relu): ReLU(inplace=True)
    )
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
 

In [4]:
from dataset import BSD500Dataset
from config import TRAIN_DIR
dataset = BSD500Dataset(
    image_dir=TRAIN_DIR,
    add_noise=True,
)
noisy, clean = dataset[0]
print(noisy.shape)
print(clean.shape)


torch.Size([3, 128, 128])
torch.Size([3, 128, 128])


In [5]:
dataset = BSD500Dataset(
    image_dir=TRAIN_DIR,
    add_noise=True,
)
noisy, clean = dataset[0]
noisy = noisy.unsqueeze(0).to(DEVICE)
output=model(noisy)
print(output.shape)

torch.Size([1, 3, 128, 128])


In [6]:
from metrics import count_parameters

print(f"trainable parameters: {count_parameters(model):,}")

trainable parameters: 181,347


In [9]:
from metrics import count_parameters

print(f"trainable parameters: {count_parameters(model2):,}")

trainable parameters: 273,699


In [11]:
from metrics import count_parameters

print(f"trainable parameters: {count_parameters(model3):,}")

trainable parameters: 615,043


In [12]:
from losses import MSELoss, HybridLoss

mse_loss=MSELoss()
hybrid_loss=HybridLoss(alpha=0.8)

In [13]:
noisy, clean = dataset[0]
clean = clean.unsqueeze(0).to(DEVICE)
prediction = clean.clone()

In [14]:
print(mse_loss(prediction,clean))
print(hybrid_loss(prediction,clean))


tensor(0., device='cuda:0')
tensor(0., device='cuda:0')


In [15]:
prediction = torch.rand_like(clean)

print(mse_loss(prediction, clean))

print(hybrid_loss(prediction, clean))

tensor(0.1156, device='cuda:0')
tensor(0.2903, device='cuda:0')


In [16]:
from dataset import BSD500Dataset
from dataset import create_dataloaders
from config import TRAIN_DIR, VAL_DIR, TEST_DIR, PATCH_SIZE, PATCHES_PER_IMAGE, BATCH_SIZE

train_dataset = BSD500Dataset(
    image_dir=TRAIN_DIR,  # Now this works
    patch_size=PATCH_SIZE,
    patches_per_image=PATCHES_PER_IMAGE,
)

val_dataset = BSD500Dataset(
    image_dir=VAL_DIR,
    patch_size=PATCH_SIZE,
    patches_per_image=PATCHES_PER_IMAGE,
)

test_dataset = BSD500Dataset(
    image_dir=TEST_DIR,
    patch_size=PATCH_SIZE,
    patches_per_image=PATCHES_PER_IMAGE,
)

train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=BATCH_SIZE,
)

In [17]:
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

63
32
63


In [19]:
from dataset import BSD500Dataset, create_dataloaders
from config import IMAGE_ROOT, TRAIN_DIR, VAL_DIR, TEST_DIR

# All parameters use defaults from config
train_dataset = BSD500Dataset(
    image_dir=TRAIN_DIR,
    # Uses PATCH_SIZE, PATCHES_PER_IMAGE, USE_PATCHES, ADD_NOISE from config
)

val_dataset = BSD500Dataset(
    image_dir=VAL_DIR,
    # Uses config defaults
)

test_dataset = BSD500Dataset(
    image_dir=TEST_DIR,
    # Uses config defaults
)

# Create dataloaders (uses BATCH_SIZE, NUM_WORKERS, PIN_MEMORY from config)
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
)